In [1]:
# %run take_split.py

In [2]:
!rm -rf cache/scoring/

In [3]:
import wandb


wandb.login()


api = wandb.Api()


entity = "loriss"
project = "linear_coder"


runs = api.runs(f"{entity}/{project}")


for run in runs:
    print(f"Deleting run: {run.name} ({run.id})")
    run.delete()


wandb: Currently logged in as: loriss to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Deleting run: MSECoder_LESSEstimator: normalize=True (cs4fj110)
Deleting run: MSECoder_LESSEstimator: normalize=True (9lmc8ugw)
Deleting run: MSECoder_LESSEstimator: normalize=True (ih2odjhq)
Deleting run: MSECoder_LESSEstimator: normalize=True (m8nlsf61)
Deleting run: MSECoder_LESSEstimator: normalize=True (qxgpdw5b)
Deleting run: CosineCoder_LESSEstimator: normalize=True (s8w84v42)


In [4]:
import os
from multiprocess import set_start_method
set_start_method("spawn")
import torch
import logging
logger = logging.getLogger("ignite.handlers.early_stopping.EarlyStopping")
logger.setLevel(logging.WARNING)


In [5]:

import torch
from scipy import stats
import numpy as np

import numpy as np
from scipy import stats
import itertools
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import os
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed


In [6]:
from load_experiment_data import (
    train_dataset_name,
    test_dataset_name,
    train_dataset_split,
    test_dataset_split,
    load_data_and_estimators,
    explanation_types,
    linear_coders
)
train_dataset, test_dataset, estimators = load_data_and_estimators()


[INFO] PyTorch version 2.7.0+cu126 available.
Repo card metadata block was not found. Setting CardData to empty.
[WARNING] Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.
[WARNING] Repo card metadata block was not found. Setting CardData to empty.


influence_estimate_path: ./results/influence/LESSEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test/estimate.parquet
dirname: ./results/influence/LESSEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test
exists: True
influence_estimate_path: ./results/influence/DataInfEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test/estimate.parquet
dirname: ./results/influence/DataInfEstimator/8192-True/pythia-31m_tulu-v2-sft-mixture_train/tulu-v2-sft-mixture_test/tulu-v2-sft-mixture_test
exists: True


In [7]:
os.path.basename(train_dataset_name)

'tulu-v2-sft-mixture'

In [8]:
for estimator in estimators:
    print(estimator.get_config_string(), estimator.get_gradient(train_dataset, train_dataset_name, train_dataset_split, 0).shape)

LESSEstimator: normalize=True torch.Size([122880])
DataInfEstimator: fast_implementation=True torch.Size([122880])


In [9]:
import logging
logging.getLogger().setLevel(logging.WARNING)
from itertools import cycle


In [10]:
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
import torch
import multiprocessing
from tqdm import tqdm
import itertools
import pandas as pd
import traceback



logging.basicConfig(level=logging.ERROR, format='%(asctime)s [%(levelname)s] %(message)s')
multiprocessing.set_start_method('spawn', force=True)
torch.manual_seed(42)


n_test = 1
test_indices = test_dataset.shuffle(seed=42).select(range(n_test))["indices"]

num_devices = torch.cuda.device_count()




In [11]:
import evaluation_worker
device_ids = itertools.cycle(range(num_devices))
results = []
import logging


logging.getLogger("wandb").setLevel(logging.ERROR)
import os
os.environ["WANDB_SILENT"] = "true"

for estimator in estimators:
    
    print("getting")

    explanations = [
        explanation_type(row.name, estimator)
        for explanation_type in explanation_types
        for _, row in estimator.influence_estimate.iloc[test_indices].iterrows()
    ]
    print("got explanations", estimator)
        
    partial_results_dir =  os.path.join("./cache/scoring/partial/",
        estimator.get_config_string(),
        os.path.basename(estimator.model_path),
        train_dataset_name,
        train_dataset_split,
        test_dataset_name,
        test_dataset_split,
        "partial")
    with ProcessPoolExecutor(max_workers=8*num_devices) as executor:
        futures = {
            executor.submit(
                evaluation_worker.process_explanation,
                partial_results_dir,
                estimator, explanation, 
                train_dataset, train_dataset_name, train_dataset_split, test_dataset, test_dataset_name, test_dataset_split, 
                linear_coders,
                next(device_ids),
                ii
            ): explanation for ii, explanation in enumerate(explanations)
        }

        with tqdm(total=len(futures), desc="Explanations", position=0) as pbar:
            for future in as_completed(futures):
                try:
                    # You can add future.result() here if you want to catch exceptions
                    future.result()  
                except Exception as e:
                    logging.error(f"A future failed: {e}\n{traceback.format_exc()}")
                    raise
                finally:
                    pbar.update(1)

getting
got explanations <influence_estimation.less_inf.LESSEstimator object at 0x7f0fe6bbfa30>


Explanations: 100%|██████████| 5/5 [01:08<00:00, 13.77s/it]


Early stopping at step 2900, best_score=-82.586601
Early stopping at step 1100, best_score=0.477572
Early stopping at step 2400, best_score=-82.580041
Early stopping at step 16000, best_score=-0.006983
Early stopping at step 1100, best_score=0.547384
Early stopping at step 16300, best_score=-0.008282
Early stopping at step 1100, best_score=-78.469963
Early stopping at step 1100, best_score=0.447253
Early stopping at step 2900, best_score=-78.436903
Early stopping at step 7400, best_score=-76.214665
Early stopping at step 1100, best_score=0.487975
Early stopping at step 8200, best_score=-76.228194
Early stopping at step 16000, best_score=-0.006983
Early stopping at step 1100, best_score=0.547384
Early stopping at step 16300, best_score=-0.008282
getting
got explanations <influence_estimation.data_inf.DataInfEstimator object at 0x7f104733d210>


Explanations:   0%|          | 0/5 [00:00<?, ?it/s][INFO] PyTorch version 2.7.0+cu126 available.
[INFO] PyTorch version 2.7.0+cu126 available.
[INFO] PyTorch version 2.7.0+cu126 available.
[INFO] PyTorch version 2.7.0+cu126 available.
[INFO] PyTorch version 2.7.0+cu126 available.
Explanations: 100%|██████████| 5/5 [00:36<00:00,  7.34s/it]


Early stopping at step 5900, best_score=-229.081808
Early stopping at step 1100, best_score=0.249123
Early stopping at step 5400, best_score=-229.090926
Early stopping at step 5900, best_score=-229.081808
Early stopping at step 1100, best_score=0.249123
Early stopping at step 5400, best_score=-229.090926
Early stopping at step 5900, best_score=-229.081808
Early stopping at step 1100, best_score=0.249123
Early stopping at step 5400, best_score=-229.090926
Early stopping at step 4800, best_score=-254.012788
Early stopping at step 1100, best_score=0.136704
Early stopping at step 5800, best_score=-253.974886
Early stopping at step 5900, best_score=-229.081808
Early stopping at step 1100, best_score=0.249123
Early stopping at step 5400, best_score=-229.090926


In [12]:
import pandas as pd

df = pd.read_parquet("./cache/scoring/partial")
# assert (df.groupby(["explanation_type", "linear_coder"]).count() == n_test).all().all()


In [13]:
df.groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('pred_gain', 'mean')].sort_values()

explanation_type                 linear_coder        estimator                                 
10 random examples with seed 42  CosineCoder         DataInfEstimator: fast_implementation=True    1.018301e+00
                                 MSECoderNNLSL2      DataInfEstimator: fast_implementation=True    1.044363e+00
                                 MSECoderElasticNet  DataInfEstimator: fast_implementation=True    1.058772e+00
                                 MSECoderLemon       DataInfEstimator: fast_implementation=True    1.058930e+00
                                 KLTCoder            DataInfEstimator: fast_implementation=True    1.059165e+00
                                 MSECoder            DataInfEstimator: fast_implementation=True    1.059799e+00
Mean-10 most average scores      CosineCoder         DataInfEstimator: fast_implementation=True    1.065274e+00
Bottom-10 least influential      CosineCoder         DataInfEstimator: fast_implementation=True    1.065274e+00
Top-10 m

In [14]:
df.groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('l1', 'mean')].sort_values()

explanation_type                 linear_coder        estimator                                 
10 random examples with seed 42  MSECoderNNLSL2      DataInfEstimator: fast_implementation=True    0.287915
Tail-10 most extreme scores      CosineCoder         DataInfEstimator: fast_implementation=True    0.600550
Top-10 most influential          CosineCoder         DataInfEstimator: fast_implementation=True    0.600550
Bottom-10 least influential      CosineCoder         DataInfEstimator: fast_implementation=True    0.600550
Mean-10 most average scores      CosineCoder         DataInfEstimator: fast_implementation=True    0.600550
10 random examples with seed 42  CosineCoder         LESSEstimator: normalize=True                 0.600550
Mean-10 most average scores      CosineCoder         LESSEstimator: normalize=True                 0.600550
Top-10 most influential          CosineCoder         LESSEstimator: normalize=True                 0.600550
Bottom-10 least influential      CosineC

In [15]:
df.groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('mse', 'mean')].sort_values()

explanation_type                 linear_coder        estimator                                 
Top-10 most influential          MSECoderNNLSL2      LESSEstimator: normalize=True                 5.375383e-14
Tail-10 most extreme scores      MSECoderNNLSL2      LESSEstimator: normalize=True                 5.375383e-14
Top-10 most influential          MSECoder            LESSEstimator: normalize=True                 2.505078e-12
Tail-10 most extreme scores      MSECoder            LESSEstimator: normalize=True                 2.505078e-12
Top-10 most influential          MSECoderElasticNet  LESSEstimator: normalize=True                 6.980758e-03
Tail-10 most extreme scores      MSECoderElasticNet  LESSEstimator: normalize=True                 6.980758e-03
                                 MSECoderLemon       LESSEstimator: normalize=True                 8.280692e-03
Top-10 most influential          MSECoderLemon       LESSEstimator: normalize=True                 8.280692e-03
        

In [16]:
df[~df["linear_coder"].str.contains("KLT")].groupby(["explanation_type", "linear_coder", "estimator"]).describe()[('pred_gain', 'mean')].sort_values(ascending=False)

explanation_type                 linear_coder        estimator                                 
Top-10 most influential          MSECoderNNLSL2      LESSEstimator: normalize=True                 1.084394e+10
Tail-10 most extreme scores      MSECoderNNLSL2      LESSEstimator: normalize=True                 1.084394e+10
                                 MSECoder            LESSEstimator: normalize=True                 1.084129e+10
Top-10 most influential          MSECoder            LESSEstimator: normalize=True                 1.084129e+10
Tail-10 most extreme scores      MSECoderElasticNet  LESSEstimator: normalize=True                 1.553427e+04
Top-10 most influential          MSECoderElasticNet  LESSEstimator: normalize=True                 1.553427e+04
                                 MSECoderLemon       LESSEstimator: normalize=True                 1.309553e+04
Tail-10 most extreme scores      MSECoderLemon       LESSEstimator: normalize=True                 1.309553e+04
Bottom-1